In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import json
import pickle
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../data/df_features.csv', parse_dates=['date'])

with open('../src/models/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

cutoff_date = '2015-12-31'
train = df[df['date'] <= cutoff_date].copy()
val   = df[df['date'] >  cutoff_date].copy()

X_train, y_train = train[feature_cols], train['sales']
X_val,   y_val   = val[feature_cols],   val['sales']

print("Data loaded. Ready for MLflow.")

Data loaded. Ready for MLflow.


In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("demand-forecasting")
print("MLflow connected. Check http://localhost:5000")

2026/06/15 15:55:59 INFO mlflow.tracking.fluent: Experiment with name 'demand-forecasting' does not exist. Creating a new experiment.


MLflow connected. Check http://localhost:5000


In [5]:
configs = [
    {'n_estimators': 1000, 'learning_rate': 0.05, 'num_leaves': 31,
     'run_name': 'lgbm_baseline'},
    {'n_estimators': 1000, 'learning_rate': 0.01, 'num_leaves': 63,
     'run_name': 'lgbm_deeper'},
    {'n_estimators': 1000, 'learning_rate': 0.05, 'num_leaves': 15,
     'run_name': 'lgbm_simpler'},
]

run_ids = {}

for config in configs:
    run_name = config.pop('run_name')

    with mlflow.start_run(run_name=run_name):
        # Train
        model = lgb.LGBMRegressor(**config, random_state=42, verbose=-1)
        model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        # Predict
        preds = np.clip(model.predict(X_val), 0, None)
        
        mse = mean_squared_error(y_val, preds)
        rmse = np.sqrt(mse)
        
        mae = np.mean(np.abs(y_val - preds))
        mape = np.mean(np.abs((y_val - preds) / (y_val + 1))) * 100

        # Log params
        mlflow.log_params(config)
        mlflow.log_param("features_count", len(feature_cols))
        mlflow.log_param("train_size", len(train))
        mlflow.log_param("val_size", len(val))
        mlflow.log_param("validation_method", "walk_forward")

        # Log metrics
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mape", mape)
        mlflow.log_metric("best_iteration", model.best_iteration_)

        # Log feature importance plot
        importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(importance['feature'][:15], importance['importance'][:15])
        ax.set_title(f"Feature Importance — {run_name}")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'../notebooks/importance_{run_name}.png')
        mlflow.log_artifact(f'../notebooks/importance_{run_name}.png')
        plt.close()

        # Log model
        mlflow.lightgbm.log_model(model, "model")

        run_id = mlflow.active_run().info.run_id
        run_ids[run_name] = run_id
        print(f"{run_name} → RMSE: {rmse:.4f} | Run ID: {run_id}")

    config['run_name'] = run_name  # restore for next iteration

2026/06/15 15:59:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 15:59:40 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lgbm_baseline → RMSE: 7.4324 | Run ID: eeced7b9bcf3485d9d80a8716e87cc15
🏃 View run lgbm_baseline at: http://localhost:5000/#/experiments/1/runs/eeced7b9bcf3485d9d80a8716e87cc15
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/06/15 16:00:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 16:00:21 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lgbm_deeper → RMSE: 7.4279 | Run ID: 52ec581a1a6344319717e6370164e823
🏃 View run lgbm_deeper at: http://localhost:5000/#/experiments/1/runs/52ec581a1a6344319717e6370164e823
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/06/15 16:00:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 16:00:33 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lgbm_simpler → RMSE: 7.4775 | Run ID: d244f47478634c02a2e0aa8197cd9743
🏃 View run lgbm_simpler at: http://localhost:5000/#/experiments/1/runs/d244f47478634c02a2e0aa8197cd9743
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [6]:
# Best model is lgbm_deeper (RMSE 7.4279)
best_run_id = run_ids['lgbm_deeper']

model_uri = f"runs:/{best_run_id}/model"
result = mlflow.register_model(model_uri, "demand-forecast-lgbm")

print(f"Model registered!")
print(f"Name: demand-forecast-lgbm")
print(f"Version: {result.version}")

Successfully registered model 'demand-forecast-lgbm'.
2026/06/15 16:02:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 52ec581a1a6344319717e6370164e823 has no artifacts at artifact path 'model', registering model based on models:/m-5859870b52d747ea82f9ce044ee83a3f instead
2026/06/15 16:02:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: demand-forecast-lgbm, version 1


Model registered!
Name: demand-forecast-lgbm
Version: 1


Created version '1' of model 'demand-forecast-lgbm'.


In [7]:
from mlflow.tracking import MlflowClient

client = MlflowClient("http://localhost:5000")

client.transition_model_version_stage(
    name="demand-forecast-lgbm",
    version=result.version,
    stage="Production"
)

print("Model transitioned to Production stage")
print("Check MLflow UI → Models tab to see it")

C:\Users\Aadya Kapoor\AppData\Local\Temp\ipykernel_1988\3546882887.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model transitioned to Production stage
Check MLflow UI → Models tab to see it


In [8]:
# This is exactly what your FastAPI app will do
loaded_model = mlflow.lightgbm.load_model(
    "models:/demand-forecast-lgbm/Production"
)

# Quick test
test_preds = np.clip(loaded_model.predict(X_val[:5]), 0, None)
print("Test predictions from registry model:")
print(test_preds.round(2))
print("\nActual values:")
print(y_val[:5].values)
print("\nModel loads from registry successfully!")

Test predictions from registry model:
[5.02 7.4  8.46 6.21 6.44]

Actual values:
[5 9 8 3 0]

Model loads from registry successfully!


In [9]:
print("""
ACTION REQUIRED — Do this now:
================================
1. Go to http://localhost:5000
2. Click 'demand-forecasting' experiment
3. You'll see 3 runs — take a screenshot
4. Click 'Models' tab — take a screenshot of registered model
5. Save both screenshots to notebooks/
   - notebooks/mlflow_experiments.png
   - notebooks/mlflow_model_registry.png

These screenshots go in your GitHub README.
They are proof of professional MLOps practice.
""")


ACTION REQUIRED — Do this now:
1. Go to http://localhost:5000
2. Click 'demand-forecasting' experiment
3. You'll see 3 runs — take a screenshot
4. Click 'Models' tab — take a screenshot of registered model
5. Save both screenshots to notebooks/
   - notebooks/mlflow_experiments.png
   - notebooks/mlflow_model_registry.png

These screenshots go in your GitHub README.
They are proof of professional MLOps practice.

